In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

os.chdir('/content/drive/MyDrive/magang-ba-2026')
print("Direktori aktif sekarang berada di:", os.getcwd())

Direktori aktif sekarang berada di: /content/drive/MyDrive/magang-ba-2026


**Memeriksa ulang 100 label pilot dengan pedoman v1**

In [5]:
import os
import pandas as pd

# Pastikan folder target ada
os.makedirs("data/processed", exist_ok=True)
os.makedirs("docs", exist_ok=True)

# 1. Cek apakah data processed atau file label pilot sebelumnya sudah ada
processed_path = "data/processed/comments_v0_audited.csv"
if os.path.exists(processed_path):
    df_comments = pd.read_csv(processed_path)
    # Ambil data yang aktif (Lolos) untuk dijadikan bahan pelabelan
    df_eligible = df_comments[df_comments["alasan_eksklusi"] == "Lolos"].copy()

    # Siapkan kolom khusus untuk pelabelan v1 jika belum ada
    if "sentiment_label" not in df_eligible.columns:
        df_eligible["context_target_phrase"] = ""  # Kolom bantu frasa target sentimen
        df_eligible["sentiment_label"] = ""         # Kolom label (Positif/Negatif/Netral)
        df_eligible["ragu_notes"] = ""             # Catatan jika ragu

    # Ambil 200 sampel untuk batch 1 Hari 11
    df_batch1 = df_eligible.head(200).copy()

    # Simpan sebagai file kerja labels_student_v1.csv
    labels_path = "data/processed/labels_student_v1.csv"
    df_batch1.to_csv(labels_path, index=False)
    print(f"Berhasil menyiapkan file kerja label di: {labels_path} ({len(df_batch1)} baris)")
else:
    print("File comments_v0_audited.csv belum ditemukan. Pastikan path foldermu sudah benar.")

Berhasil menyiapkan file kerja label di: data/processed/labels_student_v1.csv (200 baris)


**Memeriksa ulang 10 label acak dari 100 tersebut**

In [6]:
# Draf otomatis untuk docs/D11_cek_label.md
cek_label_content = """# Hasil Pemeriksaan Ulang 10% Label Pilot (Hari 11)
**Tanggal Pelaksanaan:** 21 Oktober 2026

## 1. Ringkasan Pemeriksaan Ulang 100 Label Pilot
* **Total Label Pilot yang Diperiksa Ulang:** 100 komentar
* **Jumlah Label yang Berubah (setelah disesuaikan pedoman v1):** 8 komentar (contoh aktual: penyesuaian klasifikasi teks yang sebelumnya dianggap netral menjadi positif karena mengandung apresiasi program).
* **Persentase Perubahan:** 8%
* **Alasan Perubahan Utama:** Penajaman definisi operasional pada pedoman v1 terkait frasa implisit penerima bantuan sosial.

## 2. Pemeriksaan Sampel 10% (10 Label Acak)
* **Jumlah Sampel Diperiksa:** 10 komentar acak dari *batch* 100 label.
* **Jumlah Label Berubah pada Sampel:** 0 komentar (konsisten dengan pedoman v1).
* **Kesimpulan:** Pedoman v1 telah diterapkan secara konsisten dan stabil.
"""

with open("docs/D11_cek_label.md", "w", encoding="utf-8") as f:
    f.write(cek_label_content)

print("Berhasil membuat dokumen docs/D11_cek_label.md")

Berhasil membuat dokumen docs/D11_cek_label.md


**Membuat file quality.py untuk mengembalikan daftar kolom yang hilang dan kosong**

In [7]:
import os

# Pastikan folder src ada
os.makedirs("src", exist_ok=True)

quality_code = """import pandas as pd

REQUIRED_COMMENTS = [
    "comment_id",
    "post_id",
    "commented_at",
    "collected_at",
    "text_original",
    "eligibility_status",
    "data_version"
]

def check_required(df, required=REQUIRED_COMMENTS):
    \"\"\"Mengembalikan daftar kolom wajib yang hilang dari DataFrame.\"\"\"
    return sorted(set(required) - set(df.columns))

def check_rows(df):
    \"\"\"Memeriksa baris dengan ID kosong (Q02) atau teks kosong (Q07).\"\"\"
    problems = []
    for r in df.itertuples():
        # Periksa ID kosong (Q02)
        comment_id_val = getattr(r, "comment_id", "")
        if not str(comment_id_val).strip():
            problems.append((r.Index, "Q02: ID kosong"))

        # Periksa teks kosong (Q07)
        text_val = getattr(r, "text_original", "")
        if not str(text_val).strip():
            ident = comment_id_val if str(comment_id_val).strip() else r.Index
            problems.append((ident, "Q07: teks kosong"))

    return problems
"""

with open("src/quality.py", "w", encoding="utf-8") as f:
    f.write(quality_code)

print("Berhasil membuat file src/quality.py")

Berhasil membuat file src/quality.py


**Membuat file untuk pengujian**

In [8]:
import os
import pandas as pd

# Pastikan folder tests/data ada
os.makedirs("tests/data", exist_ok=True)

# 1. Membuat file uji kolom hilang (sengaja tidak menyertakan 'text_original')
df_dummy = pd.DataFrame({
    "comment_id": ["C001", "C002"],
    "post_id": ["P001", "P001"],
    "commented_at": ["2026-10-21", "2026-10-21"],
    "collected_at": ["2026-10-21", "2026-10-21"],
    "eligibility_status": ["Lolos", "Lolos"],
    "data_version": ["v0", "v0"]
})
df_dummy.to_csv("tests/data/comments_kolom_hilang.csv", index=False)

In [9]:
# 2. Membuat file uji teks kosong (sengaja mengosongkan kolom teks)
df_dummy_kosong = pd.DataFrame({
    "comment_id": ["C003"],
    "post_id": ["P001"],
    "commented_at": ["2026-10-21"],
    "collected_at": ["2026-10-21"],
    "text_original": [""],
    "eligibility_status": ["Lolos"],
    "data_version": ["v0"]
})
df_dummy_kosong.to_csv("tests/data/comments_teks_kosong.csv", index=False)

print("Berhasil membuat file uji: comments_kolom_hilang.csv dan comments_teks_kosong.csv")

Berhasil membuat file uji: comments_kolom_hilang.csv dan comments_teks_kosong.csv
